<a href="https://colab.research.google.com/github/lyudmildimitrov1997-sudo/pppp/blob/main/Case1_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

In [ ]:
random_variable = 42

MERGE THE TWO DATASETS AND VERIFY SUPPLIER IDs

In [ ]:
data_a = pd.read_csv("supplier_security_metrics.csv")
data_b = pd.read_csv("supplier_business_context.csv")


In [ ]:
print("Unique IDs in dataset A:", data_a['supplier_id'].nunique())
print("Unique IDs in dataset B:", data_b['supplier_id'].nunique())
print("Do all IDs match exactly?", set(data_a['supplier_id']) == set(data_b['supplier_id']))

In [ ]:
data = pd.merge(left=data_a, right=data_b, on="supplier_id", how="inner")
print("Merged dataset shape:", data.shape)

EXPLORE MISSING VALUES AND CLASS DISTRIBUTION

Check percentage of suppliers with a cybersecurity incident

In [ ]:
print("\nIncident rate (Class Distribution):")
print(data["cybersecurity_incident_past_12m"].value_counts(normalize=True) * 100)

How many rows have an NaN in any column

In [ ]:
print("\nTotal rows with any missing values:", data.isnull().any(axis=1).sum())
print("Missing values details per column:")
print(data.isnull().sum()[data.isnull().sum() > 0])

Delete rows with empty values

In [ ]:
data = data.dropna(axis=0)
print("Shape after dropping rows with missing values:", data.shape)

DEFINE FEATURES AND TARGET

numerical baseline predictors

In [ ]:
feature_cols = [
    "external_attack_surface_score", "security_rating", "average_patch_delay_days",
    "mfa_coverage_pct", "security_training_coverage_pct", "prior_incidents_3y",
    "fourth_party_dependency_count", "security_questionnaire_score"
]

X = data[feature_cols]
y = data["cybersecurity_incident_past_12m"].astype(int)

Split into Train and Test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_variable)

TEST THE VARIOUS MACHINE LEARNING MODELS

Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=random_variable)
log_reg.fit(X_train, y_train)
pred_lr = log_reg.predict(X_test)
prob_lr = log_reg.predict_proba(X_test)[:, 1]

Random Forest

In [ ]:
rf_mod = RandomForestClassifier(random_state=random_variable)
rf_mod.fit(X_train, y_train)
pred_rf = rf_mod.predict(X_test)
prob_rf = rf_mod.predict_proba(X_test)[:, 1]

Gradient Boosting

In [ ]:
gb_mod = GradientBoostingClassifier(random_state=random_variable)
gb_mod.fit(X_train, y_train)
pred_gb = gb_mod.predict(X_test)
prob_gb = gb_mod.predict_proba(X_test)[:, 1]

COMPARE MODELS USING APPROPRIATE METRICS

In [ ]:
print("\n" + "="*40)
print("LOGISTIC REGRESSION PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_lr))
print("F1-Score: ", f1_score(y_test, pred_lr))
print("ROC-AUC:  ", roc_auc_score(y_test, prob_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_lr))

print("\n" + "="*40)
print("RANDOM FOREST PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_rf))
print("F1-Score: ", f1_score(y_test, pred_rf))
print("ROC-AUC:  ", roc_auc_score(y_test, prob_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_rf))

print("\n" + "="*40)
print("GRADIENT BOOSTING PERFORMANCE:")
print("Accuracy: ", accuracy_score(y_test, pred_gb))
print("F1-Score: ", f1_score(y_test, pred_gb))
print("ROC-AUC:  ", roc_auc_score(y_test, prob_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_gb))